In [241]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from keras import layers, Model
from keras.callbacks import EarlyStopping

In [242]:
from pathlib import Path
DATA_DF = Path("..") / "data" / "Pain Data.xlsx"   
df = pd.read_excel(DATA_DF)

In [243]:
df.head()


,id,Age,Gender (Biological),What region is your family origin from?,The district you live in (For weather quality),How many hours do you sleep per day on average?,How many hours did you sleep before this test?,Have you had any problems falling asleep or staying asleep? (Yes No),How have you been feeling in general lately?,What do you think about your stress level in your daily life?,...,At what times of the day do you feel your pain?,Do you suffer from fever or chills at night?,Physical Factor Table [I am asked to do too much work],Physical Factor Table [I need to work out a lot],Physical Factor Table [I enjoy my name],Physical Factor Table [I have enough time to do my name],Physical Factor Table [I am asked to perform unrealistic tasks],Physical Factor Table [I do a monotonous job],Physical Factor Table [I lift heavy loads],Physical Factor Table [I lift light loads]
0,S001,24.0,Female,NaN,ESENYURT DISTRICT,7,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S002,20.0,Male,Eastern Anatolia Region,BAKIRKÖY DISTRICT,6-8 hours,7-8 hours,No,Stressed,High,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,S003,23.0,Male,Central Anatolia Region,ESENYURT DISTRICT,4-6 hours,5-6 hours,No,Tired,Medium,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S004,23.0,Male,Aegean Region,BAKIRKÖY DISTRICT,6-8 hours,5-6 hours,Yes,Tense,Medium,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,S005,24.0,Female,Southeastern Anatolia Region,BAYRAMPAŞA DISTRICT,6-8 hours,5-6 hours,No,Stressed,High,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [244]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Columns: 108 entries, id to Physical Factor Table [I lift light loads]
dtypes: float64(7), str(101)
memory usage: 144.4 KB


In [245]:
pain_cols = [col for col in df.columns if 'pain' in col.lower()]

for i, col in enumerate(pain_cols):
    print(i, col)

0 Do you have any chronic pain conditions?
1 What is your pain type?
2 Rate the severity of your pain (Likert scale) [How severe is your pain now?]
3 Rate the severity of your pain (Likert scale) [How bothersome is your pain right now?]
4 Rate the severity of your pain (Likert scale) [How much does your pain currently affect your daily activities?]
5 Rate the severity of your pain (Likert scale) [How unbearable is your pain right now?]
6 Rate the severity of your pain (Likert scale) [How widespread is your pain now?]
7 How does your pain change over time? Tick ​​the most appropriate group.
8 How much does your menstrual pain interfere with your daily life activities? (Likert scale: 1 - Not at all hindering, 5 - Very much hindering)
9 Do you use medication for your menstrual pain?
10 Are you doing anything to relieve menstrual pain?
11 Are there any other symptoms accompanying menstrual pain?
12 Does the severity of your menstrual pain change during different periods? (Yes No)
13 What d

Based off these pain columns, the column that would be best fit is "Rate the severity of your pain (Likert scale) [How severe is your pain now?]".

In [246]:
exercise_col = [exc for exc in df.columns if 'exercise' in exc.lower()]

for i, exc in enumerate(exercise_col):
    print(i, exc)

0 How would you rate yourself based on your exercise frequency?
1 Are there any factors that initiate back pain? (Poor posture, lack of exercise, trauma, etc.)
2 Are you doing anything to relieve your back pain? (Massage, exercise, using painkillers, etc.)


Since this is an exercise app, I checked the Excel dataset to see if any features relate to exercise. The closest match is the question: “How would you rate yourself based on your exercise frequency?” This feature will allow the model to learn whether a user’s recovery tends to be faster or slower depending on their general activity level or inactivity.

I will also using age, as we know that is highly correlated with recovery length. 

In [247]:
col = [
    'Age',
    'Rate the severity of your pain (Likert scale) [How severe is your pain now?]',
    'How would you rate yourself based on your exercise frequency?'
]

df[col].isnull().sum()

Age                                                                              1
Rate the severity of your pain (Likert scale) [How severe is your pain now?]    17
How would you rate yourself based on your exercise frequency?                    0
dtype: int64

In [248]:
df = df.dropna(subset=["Age"])

In [249]:
severity_col = 'Rate the severity of your pain (Likert scale) [How severe is your pain now?]'
df.loc[df[severity_col].isnull(), "What is your pain type?"].value_counts(dropna=False)

What is your pain type?
No pain    14
NaN         3
Name: count, dtype: int64

Since these user pain responses were not helpful for the data, i will be dropping them as well. 

In [250]:
no_pain = df['What is your pain type?'] == 'No pain'
df.loc[no_pain & df[severity_col].isnull(), severity_col] = 'Not severe at all'
df = df.dropna(subset=[severity_col]) 

In [251]:
df[severity_col].unique()

<ArrowStringArray>
['Severe', 'Not severe at all', 'Moderate', 'Mild', 'Very severe']
Length: 5, dtype: str

In [252]:
exercise_col = 'How would you rate yourself based on your exercise frequency?'
df[exercise_col].unique()

<ArrowStringArray>
[ 'I never exercise regularly.',        'I exercise sometimes.',
    'I do not exercise at all.', 'I exercise most of the time.',
           'I always exercise.']
Length: 5, dtype: str

In [253]:
severity_map = {'Not severe at all': 1, 'Mild': 3, 'Moderate': 5, 'Severe':7, 'Very severe': 9}
exercise_map = {'I never exercise regularly.': 0, 'I do not exercise at all.': 0, 'I exercise sometimes.': 1, 'I exercise most of the time.': 2, 'I always exercise.': 3}

df['pain_severity'] = df[severity_col].map(severity_map)
df['exercise_level'] = df[exercise_col].map(exercise_map)

I mapped the levels for ‘Rate the severity of your pain’ and ‘Rate your exercise frequency’ so the model can learn how pain trends relate to activity levels. This helps the system evaluate whether a user’s recovery is progressing normally or if intervention is needed.

In [254]:
X = df[['Age', 'pain_severity', 'exercise_level']]
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

In [255]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train.shape)
print(X_test.shape)

(76, 3)
(19, 3)


In [256]:
class AnomalyDetector(Model):
    def __init__(self, input_dim):
        super(AnomalyDetector, self).__init__()
       
        self.encoder = tf.keras.Sequential([
            layers.Dense(4, activation="relu"),
            layers.Dense(2, activation="relu") 
        ])
       
        self.decoder = tf.keras.Sequential([
            layers.Dense(4, activation="relu"),
            layers.Dense(input_dim, activation="sigmoid") 
        ])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


tf.keras.utils.set_random_seed(42)
autoencoder = AnomalyDetector(input_dim=X_train_scaled.shape[1])
autoencoder.compile(optimizer='adam', loss='mae') 

In [257]:
es = EarlyStopping( monitor='val_loss', patience=10, restore_best_weights=True)

In [258]:
history = autoencoder.fit(X_train_scaled, X_train_scaled, epochs=500, batch_size=8, validation_data=(X_test_scaled, X_test_scaled), callbacks=[es])

Epoch 1/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.3084 - val_loss: 0.3066
Epoch 2/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3037 - val_loss: 0.3032
Epoch 3/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2998 - val_loss: 0.3009
Epoch 4/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2973 - val_loss: 0.2991
Epoch 5/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2956 - val_loss: 0.2975
Epoch 6/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2941 - val_loss: 0.2960
Epoch 7/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2928 - val_loss: 0.2947
Epoch 8/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2915 - val_loss: 0.2934
Epoch 9/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2902 - val_loss: 0.2921
Epoch 10/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2889 - val_loss: 0.2908
Epoch 11/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2877 - val_loss: 0.2896
Epoch 12/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.

In [259]:
train_predict = autoencoder.predict(X_train_scaled)
train_loss = np.mean(np.abs(train_predict - X_train_scaled), axis=1)

threshold = np.percentile(train_loss, 95)
print(f'Threshold:', threshold)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Threshold: 0.1630085990336782


I calculated a threshold by measuring how much the model typically “messes up” when reconstructing normal data. I set the threshold at the 95th percentile of reconstruction errors, meaning any new sample with an error higher than what 95% of normal data produces will be flagged as an anomaly.

In [260]:
import joblib

autoencoder.save_weights("recovery_autoencoder.weights.h5")   # the learned weights
joblib.dump(scaler, "autoencoder_scaler.joblib")
joblib.dump(threshold, "autoencoder_threshold.joblib")

['autoencoder_threshold.joblib']

Due to a smart control app block, my code is trained in TensorFlow/Keras, but served with NumPy.